In [3]:
# Cell 1: setup
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TOP_POPULAR_TRACKS = 12000
INCLUDE_TEST_CANDIDATES = True

AE_HIDDEN_DIM = 256
AE_LATENT_DIM = 64
AE_DROPOUT = 0.15
AE_EPOCHS = 10
AE_BATCH_SIZE = 128
AE_LR = 1e-3
AE_WEIGHT_DECAY = 1e-5
AE_VALID_FRACTION = 0.10

MODEL_DIR = Path("SavedModels/autoencoder")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "track_autoencoder.pt"
TRACK_PRIORS_PATH = MODEL_DIR / "track_popularity.csv"

print("Device:", DEVICE)
print("Model path:", MODEL_PATH)


Device: cpu
Model path: SavedModels\autoencoder\track_autoencoder.pt


d:\Direct\Programs\Languages\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Cell 2: load data and keep direct track ratings only

track_df = pd.read_csv("Assets/CSV/track_data.csv")
train_df = pd.read_csv("Assets/CSV/train_data.csv")
test_df = pd.read_csv("Assets/CSV/test_data.csv")

all_track_ids = set(track_df["TrackID"].dropna().astype(int).tolist())

train_df["ItemID_str"] = train_df["ItemID"].astype(int).astype(str)
track_id_str = set(str(t) for t in all_track_ids)

train_track_rows = train_df[train_df["ItemID_str"].isin(track_id_str)].copy()

print("Tracks in metadata        :", len(all_track_ids))
print("Training rows             :", len(train_df))
print("Direct track-rating rows  :", len(train_track_rows))
print("Unique test candidate tracks:", test_df["TrackID"].nunique())

Tracks in metadata        : 224041
Training rows             : 12403575
Direct track-rating rows  : 5480041
Unique test candidate tracks: 48832


In [5]:
# Cell 3: build AE vocabulary and sparse matrix

test_track_ids = set(test_df["TrackID"].astype(int).unique().tolist())
popular_track_ids = set(
    train_track_rows["ItemID"].value_counts().head(TOP_POPULAR_TRACKS).index.astype(int).tolist()
)

ae_track_ids = set(popular_track_ids)
if INCLUDE_TEST_CANDIDATES:
    ae_track_ids |= test_track_ids

ae_track_ids = sorted(ae_track_ids)
ae_track_id_str = set(str(t) for t in ae_track_ids)

train_track_rows_ae = train_track_rows[train_track_rows["ItemID_str"].isin(ae_track_id_str)].copy()

ae_user_ids = sorted(train_track_rows_ae["UserID"].astype(int).unique().tolist())
user_to_idx = {uid: i for i, uid in enumerate(ae_user_ids)}
track_to_idx = {tid: j for j, tid in enumerate(ae_track_ids)}

row_idx = train_track_rows_ae["UserID"].astype(int).map(user_to_idx).to_numpy()
col_idx = train_track_rows_ae["ItemID"].astype(int).map(track_to_idx).to_numpy()
values = (train_track_rows_ae["Rating"].astype(np.float32) / 100.0).to_numpy()

ratings_csr = csr_matrix(
    (values, (row_idx, col_idx)),
    shape=(len(ae_user_ids), len(ae_track_ids)),
    dtype=np.float32,
)

mask_csr = ratings_csr.copy()
mask_csr.data = np.ones_like(mask_csr.data, dtype=np.float32)

rng = np.random.default_rng(RANDOM_SEED)
observed_row_ids = np.where(ratings_csr.getnnz(axis=1) > 0)[0]
rng.shuffle(observed_row_ids)

valid_cut = max(1, int(AE_VALID_FRACTION * len(observed_row_ids)))
ae_valid_rows = np.sort(observed_row_ids[:valid_cut])
ae_train_rows = np.sort(observed_row_ids[valid_cut:])

print("AE users              :", len(ae_user_ids))
print("AE track vocabulary   :", len(ae_track_ids))
print("Observed ratings      :", ratings_csr.nnz)
print("Density               :", ratings_csr.nnz / (ratings_csr.shape[0] * ratings_csr.shape[1]))
print("Train rows            :", len(ae_train_rows))
print("Valid rows            :", len(ae_valid_rows))

AE users              : 34361
AE track vocabulary   : 49929
Observed ratings      : 3516736
Density               : 0.002049845386543009
Train rows            : 30925
Valid rows            : 3436


In [6]:
# Cell 4: helpers + model

def iter_dense_batches(ratings_matrix, mask_matrix, row_ids, batch_size, shuffle=True):
    row_ids = np.array(row_ids, copy=True)
    if shuffle:
        np.random.shuffle(row_ids)

    for start in range(0, len(row_ids), batch_size):
        batch_ids = row_ids[start:start + batch_size]
        batch_x = torch.tensor(ratings_matrix[batch_ids].toarray(), dtype=torch.float32, device=DEVICE)
        batch_mask = torch.tensor(mask_matrix[batch_ids].toarray(), dtype=torch.float32, device=DEVICE)
        yield batch_ids, batch_x, batch_mask


class TrackAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, latent_dim=64, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def masked_mse(pred, target, mask):
    diff = (pred - target) * mask
    denom = torch.clamp(mask.sum(), min=1.0)
    return diff.pow(2).sum() / denom


model = TrackAutoencoder(
    input_dim=len(ae_track_ids),
    hidden_dim=AE_HIDDEN_DIM,
    latent_dim=AE_LATENT_DIM,
    dropout=AE_DROPOUT,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=AE_LR,
    weight_decay=AE_WEIGHT_DECAY,
)

print(model)


TrackAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=49929, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=49929, bias=True)
    (3): Sigmoid()
  )
)


In [7]:
# Cell 5: train

history = []

for epoch in range(1, AE_EPOCHS + 1):
    model.train()
    train_losses = []

    for _, batch_x, batch_mask in iter_dense_batches(
        ratings_csr,
        mask_csr,
        ae_train_rows,
        AE_BATCH_SIZE,
        shuffle=True,
    ):
        optimizer.zero_grad()
        pred = model(batch_x)
        loss = masked_mse(pred, batch_x, batch_mask)
        loss.backward()
        optimizer.step()
        train_losses.append(float(loss.item()))

    model.eval()
    valid_losses = []
    with torch.no_grad():
        for _, batch_x, batch_mask in iter_dense_batches(
            ratings_csr,
            mask_csr,
            ae_valid_rows,
            AE_BATCH_SIZE,
            shuffle=False,
        ):
            pred = model(batch_x)
            loss = masked_mse(pred, batch_x, batch_mask)
            valid_losses.append(float(loss.item()))

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)) if train_losses else np.nan,
        "valid_loss": float(np.mean(valid_losses)) if valid_losses else np.nan,
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.6f} | "
        f"valid_loss={row['valid_loss']:.6f}"
    )

history_df = pd.DataFrame(history)
history_df

Epoch 01 | train_loss=0.125281 | valid_loss=0.116553
Epoch 02 | train_loss=0.107924 | valid_loss=0.110680
Epoch 03 | train_loss=0.094776 | valid_loss=0.107701
Epoch 04 | train_loss=0.087754 | valid_loss=0.105956
Epoch 05 | train_loss=0.083091 | valid_loss=0.106434
Epoch 06 | train_loss=0.080405 | valid_loss=0.106496
Epoch 07 | train_loss=0.077327 | valid_loss=0.105883
Epoch 08 | train_loss=0.074909 | valid_loss=0.106208
Epoch 09 | train_loss=0.073390 | valid_loss=0.106524
Epoch 10 | train_loss=0.072053 | valid_loss=0.106820


,epoch,train_loss,valid_loss
0,1,0.125281,0.116553
1,2,0.107924,0.110680
2,3,0.094776,0.107701
3,4,0.087754,0.105956
4,5,0.083091,0.106434
5,6,0.080405,0.106496
6,7,0.077327,0.105883
7,8,0.074909,0.106208
8,9,0.073390,0.106524
9,10,0.072053,0.106820


In [8]:
# Cell 6: build track popularity priors and save everything needed later

overall_mean = float(train_df["Rating"].mean())

track_stats = (
    train_track_rows.groupby("ItemID")["Rating"]
    .agg(["mean", "count"])
    .reset_index()
)
track_stats.columns = ["TrackID", "global_mean", "rating_count"]
track_stats["TrackID"] = track_stats["TrackID"].astype(int)

C = float(track_stats["rating_count"].median())
track_stats["bayesian_avg"] = (
    (track_stats["rating_count"] * track_stats["global_mean"] + C * overall_mean)
    / (track_stats["rating_count"] + C)
)

track_stats[["TrackID", "bayesian_avg", "rating_count"]].to_csv(TRACK_PRIORS_PATH, index=False)

checkpoint = {
    "state_dict": model.state_dict(),
    "model_config": {
        "input_dim": len(ae_track_ids),
        "hidden_dim": AE_HIDDEN_DIM,
        "latent_dim": AE_LATENT_DIM,
        "dropout": AE_DROPOUT,
    },
    "training_config": {
        "top_popular_tracks": TOP_POPULAR_TRACKS,
        "include_test_candidates": INCLUDE_TEST_CANDIDATES,
        "epochs": AE_EPOCHS,
        "batch_size": AE_BATCH_SIZE,
        "lr": AE_LR,
        "weight_decay": AE_WEIGHT_DECAY,
        "random_seed": RANDOM_SEED,
    },
    "track_ids": ae_track_ids,
    "user_ids": ae_user_ids,
    "history": history,
    "overall_mean": overall_mean,
}

torch.save(checkpoint, MODEL_PATH)

print("Saved model checkpoint:", MODEL_PATH)
print("Saved track priors    :", TRACK_PRIORS_PATH)


Saved model checkpoint: SavedModels\autoencoder\track_autoencoder.pt
Saved track priors    : SavedModels\autoencoder\track_popularity.csv


In [10]:
# Cell 8: pure autoencoder submission

OUTPUT_CSV = "submission_autoencoder_only.csv"

# Load saved checkpoint if you want this cell to work independently
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

inference_model = TrackAutoencoder(
    input_dim=checkpoint["model_config"]["input_dim"],
    hidden_dim=checkpoint["model_config"]["hidden_dim"],
    latent_dim=checkpoint["model_config"]["latent_dim"],
    dropout=checkpoint["model_config"]["dropout"],
).to(DEVICE)
inference_model.load_state_dict(checkpoint["state_dict"])
inference_model.eval()

saved_track_ids = checkpoint["track_ids"]
saved_user_ids = checkpoint["user_ids"]

track_to_idx_infer = {tid: j for j, tid in enumerate(saved_track_ids)}
user_to_idx_infer = {uid: i for i, uid in enumerate(saved_user_ids)}

# Rebuild the same sparse matrix shape used during training
row_idx_full = train_track_rows_ae["UserID"].astype(int).map(user_to_idx_infer).to_numpy()
col_idx_full = train_track_rows_ae["ItemID"].astype(int).map(track_to_idx_infer).to_numpy()
val_full = (train_track_rows_ae["Rating"].astype(np.float32) / 100.0).to_numpy()

ratings_csr_infer = csr_matrix(
    (val_full, (row_idx_full, col_idx_full)),
    shape=(len(saved_user_ids), len(saved_track_ids)),
    dtype=np.float32,
)

mask_csr_infer = ratings_csr_infer.copy()
mask_csr_infer.data = np.ones_like(mask_csr_infer.data, dtype=np.float32)

# Popularity fallback for any unseen user/track edge cases
track_priors_df = pd.read_csv(TRACK_PRIORS_PATH)
track_prior_lookup = dict(
    zip(track_priors_df["TrackID"].astype(int), track_priors_df["bayesian_avg"].astype(float))
)
overall_mean_saved = float(checkpoint["overall_mean"])

test_candidate_map = (
    test_df.groupby("UserID")["TrackID"]
    .apply(list)
    .to_dict()
)

ae_score_lookup_test = {}

with torch.no_grad():
    all_user_rows = np.arange(ratings_csr_infer.shape[0])

    for batch_ids, batch_x, _ in iter_dense_batches(
        ratings_csr_infer,
        mask_csr_infer,
        all_user_rows,
        AE_BATCH_SIZE,
        shuffle=False,
    ):
        pred = inference_model(batch_x).cpu().numpy() * 100.0

        for local_i, user_row in enumerate(batch_ids):
            uid = saved_user_ids[int(user_row)]
            for tid in test_candidate_map.get(uid, []):
                tid = int(tid)
                col = track_to_idx_infer.get(tid)
                if col is not None:
                    ae_score_lookup_test[(int(uid), tid)] = float(pred[local_i, col])

predictions = []

for user_id, group in tqdm(
    test_df.groupby("UserID"),
    total=test_df["UserID"].nunique(),
    desc="Generating AE-only submission",
):
    scored = []

    for tid in group["TrackID"].astype(int).tolist():
        score = ae_score_lookup_test.get(
            (int(user_id), int(tid)),
            track_prior_lookup.get(int(tid), overall_mean_saved)
        )
        scored.append((tid, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    for i, (tid, _) in enumerate(scored):
        predictions.append({
            "TrackID": f"{int(user_id)}_{int(tid)}",
            "Predictor": 1 if i < 3 else 0,
        })

submission_autoencoder_only = pd.DataFrame(predictions)
submission_autoencoder_only.to_csv(OUTPUT_CSV, index=False)

# sanity check
check_df = submission_autoencoder_only.copy()
check_df["uid"] = check_df["TrackID"].str.split("_").str[0]
bad_users = (check_df.groupby("uid")["Predictor"].sum() != 3).sum()

print("Saved:", OUTPUT_CSV)
print("Invalid users:", int(bad_users))
print("Rows:", len(submission_autoencoder_only))
print("Positive labels:", int(submission_autoencoder_only["Predictor"].sum()))
submission_autoencoder_only.head(12)


Generating AE-only submission: 100%|██████████| 20000/20000 [00:01<00:00, 11548.40it/s]


Saved: submission_autoencoder_only.csv
Invalid users: 0
Rows: 120000
Positive labels: 60000


,TrackID,Predictor
0,199810_208019,1
1,199810_18515,1
2,199810_242681,1
3,199810_105760,0
4,199810_74139,0
5,199810_9903,0
6,199812_130023,1
7,199812_142408,1
8,199812_29189,1
9,199812_223706,0


In [9]:
# Cell 7: optional quick load test

loaded = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)

loaded_model = TrackAutoencoder(
    input_dim=loaded["model_config"]["input_dim"],
    hidden_dim=loaded["model_config"]["hidden_dim"],
    latent_dim=loaded["model_config"]["latent_dim"],
    dropout=loaded["model_config"]["dropout"],
)
loaded_model.load_state_dict(loaded["state_dict"])
loaded_model.eval()

print("Reloaded OK")
print("Saved track count:", len(loaded["track_ids"]))
print("Saved user count :", len(loaded["user_ids"]))


Reloaded OK
Saved track count: 49929
Saved user count : 34361
